# Leaf Segmentation Evaluation - U-Net (ResNet101)
### Evaluating the classic Local Convolution baseline architecture.

---

### Overview

#### Evaluating the classic Local Convolution baseline architecture.

### 5 Canonical Classes:
```
0: Background
1: Epidermis
2: Vascular_Region
3: Mesophyll
4: Air_Space
```

### Key Features:
- Config-based dataset system (add new labs without changing code)
- Patch-based training (memory efficient for large CT images)
- Class-index masks (0–5) with ignore_index=255
- Focal + Jaccard hybrid loss (handles class imbalance)
- NERSC-optimized (A100 GPU, fast DataLoader)

### Pipeline:
1. Setup & GPU check
2. Load dataset config
3. Dataset + PatchDataset classes
4. DataLoader
5. Model setup
6. Training loop (with validation + checkpointing)
7. Evaluate best model
8. Inference

---
    

## Step 1: GPU Checking

In [1]:
import os
os.environ['TORCH_HOME'] = '/pscratch/sd/w/worasit/torch_cache'
os.environ['HF_HUB_OFFLINE'] = '1'

# CRITICAL: Set CUDA_VISIBLE_DEVICES *before* importing torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [2]:
# Check GPU
!nvidia-smi

Tue Mar 10 21:40:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:03:00.0 Off |                    0 |
| N/A   31C    P0             75W /  500W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

---
## Step 2: Install Dependencies
Run once only. Restart kernel affter if needed.

In [3]:
!pip install segmentation-models-pytorch albumentations --quiet

---
## Step 3: Import Libraries

In [4]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast
from tqdm import tqdm

import albumentations as A
import segmentation_models_pytorch as smp

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"SMP: {smp.__version__}")

/pscratch/sd/w/worasit/leafseg_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported
PyTorch: 2.8.0+cu128
SMP: 0.5.0


---
## Step 4 — Configuration

### When Adding a New Dataset:

**A. Check your new data first:**
```bash
# Run in NERSC terminal to inspect new dataset
python3 << 'EOF'
import numpy as np
from PIL import Image
import os

img_dir  = "/path/to/new/images"
mask_dir = "/path/to/new/masks"

# 1. Check image sizes and channels
imgs = sorted([f for f in os.listdir(img_dir) if not f.startswith('.')])
sizes = {}
rgb_count = 0
for f in imgs:
    arr = np.array(Image.open(os.path.join(img_dir, f)))
    sizes[arr.shape] = sizes.get(arr.shape, 0) + 1
    if arr.ndim == 3: rgb_count += 1
print("Image sizes:", sizes)
print("RGB images:", rgb_count)

# 2. Check mask pixel values
all_vals = set()
masks = sorted([f for f in os.listdir(mask_dir) if not f.startswith('.')])
for f in masks[:10]:
    arr = np.array(Image.open(os.path.join(mask_dir, f)))
    if arr.ndim == 3: arr = arr[:,:,0]
    all_vals.update(np.unique(arr).tolist())
print("Mask values:", sorted(all_vals))
EOF
```

**B. Create config file for new dataset:**
```json
{
  "name": "new_lab_name",
  "image_dir": "/pscratch/sd/w/worasit/datasets/new_lab/images",
  "mask_dir":  "/pscratch/sd/w/worasit/datasets/new_lab/masks",
  "num_classes": 6,
  "ignore_index": 254,
  "class_names": ["background","epidermis","mesophyll","air","vein","bundle_sheath"],
  "mapping": {
    "THEIR_BG_VALUE":  0,
    "THEIR_EPI_VALUE": 1,
    "THEIR_MESO_VALUE":2,
    "THEIR_AIR_VALUE": 3,
    "THEIR_VEIN_VALUE":4
  }
}
```

**C. Add one line to CONFIG_PATHS below.**

**D. Check if PATCH_SIZE needs adjusting:**
```
If new dataset min_height < current PATCH_SIZE -> reduce PATCH_SIZE
Rule: PATCH_SIZE <= min(min_height, min_width) of ALL datasets combined
```

In [5]:
# ============================================================
# PATHS
# ============================================================
SCRATCH = "/pscratch/sd/w/worasit"

# --- ADD/REMOVE DATASETS HERE ---
CONFIG_PATHS = [
    f"{SCRATCH}/configs/devin1_no_bse.json",        # 272 images
    f"{SCRATCH}/configs/devin1_with_bse.json",      # 56 images
    f"{SCRATCH}/configs/devin2.json",               # 24 images
    f"{SCRATCH}/configs/devin3.json",               # 7 images
    f"{SCRATCH}/configs/jg_laca.json",              # 8 images    
    f"{SCRATCH}/configs/jg_mag.json",               # 12 images        
    f"{SCRATCH}/configs/lf_arab.json",              # 55 images   
    f"{SCRATCH}/configs/oak_ce.json",               # 18 images    
    f"{SCRATCH}/configs/oak_cf.json",               # 18 images        
    f"{SCRATCH}/configs/oak_cr.json",               # 18 images        
    f"{SCRATCH}/configs/oak_ob.json",               # 18 images        
    f"{SCRATCH}/configs/oak_ru.json",               # 18 images    
    f"{SCRATCH}/configs/oak_su.json",               # 18 images    
    f"{SCRATCH}/configs/olive_d4.json",             # 6 images
    f"{SCRATCH}/configs/olive_d5.json",             # 6 images
    f"{SCRATCH}/configs/olive_r1.json",             # 6 images
    f"{SCRATCH}/configs/olive_r2.json",             # 2 images
    f"{SCRATCH}/configs/olive_w4.json",             # 6 images            
    f"{SCRATCH}/configs/olive_w5.json",             # 6 images 
    f"{SCRATCH}/configs/st_pinus_aa.json",          # 6 images
    f"{SCRATCH}/configs/st_pinus_lo1.json",         # 6 images
    f"{SCRATCH}/configs/st_pinus_lo2.json",         # 6 images    
    f"{SCRATCH}/configs/st_pinus_palus.json",       # 6 images    
    f"{SCRATCH}/configs/st_pinus_pb.json",          # 6 images    
    f"{SCRATCH}/configs/st_pinus_pc.json",          # 6 images    
    f"{SCRATCH}/configs/st_pinus_pd.json",          # 6 images    
    f"{SCRATCH}/configs/st_pinus_pe1.json",         # 6 images    
    f"{SCRATCH}/configs/st_pinus_pe2.json",         # 7 images        
    f"{SCRATCH}/configs/st_pinus_pf1.json",         # 6 images    
    f"{SCRATCH}/configs/st_pinus_pf3.json",         # 5 images        
    f"{SCRATCH}/configs/st_pinus_pg.json",          # 6 images        
    f"{SCRATCH}/configs/st_pinus_ph.json",          # 6 images        
    f"{SCRATCH}/configs/st_pinus_pinaster.json",    # 6 images        
    f"{SCRATCH}/configs/st_pinus_pinea.json",       # 5 images        
    f"{SCRATCH}/configs/st_pinus_pj1.json",         # 6 images    
    f"{SCRATCH}/configs/st_pinus_pj2.json",         # 6 images        
    f"{SCRATCH}/configs/st_pinus_pk.json",          # 6 images        
    f"{SCRATCH}/configs/st_pinus_pm2.json",         # 6 images        
    f"{SCRATCH}/configs/st_pinus_pm5.json",         # 5 images       
    f"{SCRATCH}/configs/st_pinus_pn1.json",         # 5 images    
    f"{SCRATCH}/configs/st_pinus_pn2.json",         # 6 images        
    f"{SCRATCH}/configs/st_pinus_ppd2.json",        # 6 images        
    f"{SCRATCH}/configs/st_pinus_ppg1.json",        # 6 images        
    f"{SCRATCH}/configs/st_pinus_ppg2.json",        # 6 images          
    f"{SCRATCH}/configs/st_pinus_pr1.json",         # 6 images       
    f"{SCRATCH}/configs/st_pinus_pr2.json",         # 6 images    
    f"{SCRATCH}/configs/st_pinus_pse1.json",        # 5 images        
    f"{SCRATCH}/configs/st_pinus_pth5.json",        # 6 images        
    f"{SCRATCH}/configs/st_pinus_tm10.json",        # 4 images           
]

OUTPUT_DIR = f"{SCRATCH}/outputs"
MODEL_DIR = f"{OUTPUT_DIR}/models_Unet_ResNet101"
LOG_DIR = f"{OUTPUT_DIR}/logs_Unet_ResNet101"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

# ============================================================
# TRAINING PARAMETERS
# ============================================================

# PATCH_SIZE: must be <= minimum image height/width across ALL datasets
# devin1 min height = 323 -> use 256 (safe, power of 2)
# If you add a dataset with smaller images -> reduce this value
PATCH_SIZE  = 320
STRIDE      = 160    # 50% overlap between patches

BATCH_SIZE  = 32     # 16, 32 still safe
NUM_EPOCHS  = 200
LR          = 5e-5
VAL_FRAC    = 0.1    # 90% train, 10% val
PATIENCE    = 20     # Early stopping patience
NUM_CLASSES = 5      # Always 5 canonical classes

# IGNORE_INDEX: pixel value to SKIP during training
# MUST NOT conflict with any grayscale value in your masks!
# devin1 mask values = [0, 85, 170, 180, 255] -> use 254 (safe!)
# If a future dataset uses 254 in masks → change to another unused value
IGNORE_INDEX = 254

# ============================================================

# Load and verify all configs
configs = []
for path in CONFIG_PATHS:
    assert os.path.exists(path), f"Config not found: {path}"
    with open(path) as f:
        cfg = json.load(f)
    configs.append(cfg)
    print(f"Config: {cfg['name']}")
    print(f"  image_dir: {cfg['image_dir']}")
    print(f"  mask_dir:  {cfg['mask_dir']}")
    print(f"  mapping:   {cfg['mapping']}")

print(f"\nTotal configs: {len(configs)}")
print(f"PATCH_SIZE:    {PATCH_SIZE}")
print(f"IGNORE_INDEX:  {IGNORE_INDEX}")
print(f"Output dir:    {OUTPUT_DIR}")

Config: devin1_no_bse
  image_dir: /pscratch/sd/w/worasit/datasets/devin1/images
  mask_dir:  /pscratch/sd/w/worasit/datasets/devin1/masks
  mapping:   {'170': 0, '85': 1, '180': 2, '0': 3, '255': 4}
Config: devin1_with_bse
  image_dir: /pscratch/sd/w/worasit/datasets/devin1/images
  mask_dir:  /pscratch/sd/w/worasit/datasets/devin1/masks
  mapping:   {'0': 3, '85': 1, '152': 2, '170': 0, '180': 2, '255': 4}
Config: devin2
  image_dir: /pscratch/sd/w/worasit/datasets/devin2/images
  mask_dir:  /pscratch/sd/w/worasit/datasets/devin2/masks
  mapping:   {'0': 3, '85': 1, '95': 1, '152': 2, '170': 0, '180': 2, '255': 4}
Config: devin3
  image_dir: /pscratch/sd/w/worasit/datasets/devin3/images
  mask_dir:  /pscratch/sd/w/worasit/datasets/devin3/masks
  mapping:   {'0': 4, '85': 0, '103': 2, '170': 1, '255': 3}
Config: jg_laca
  image_dir: /pscratch/sd/w/worasit/datasets/jg_laca/images
  mask_dir:  /pscratch/sd/w/worasit/datasets/jg_laca/masks
  mapping:   {'0': 4, '100': 2, '130': 1, '170':

---
## Step 5: Dataset Class

### Handles:
- Grayscale AND RGB images (auto-converts)
- Any image size (PatchDataset handles patches)
- Any grayscale mapping (from config file)
- Missing classes (e.g. no BSE in some datasets)

In [6]:
class LeafDataset(Dataset):
    """
    Loads full CT images and masks from a config file.
    Converts grayscale mask values -> canonical class indices (0-5).
    Handles both grayscale and RGB images automatically.
    """

    def __init__(self, config_path):
        with open(config_path, "r") as f:
            self.cfg = json.load(f)

        self.name      = self.cfg["name"]
        self.image_dir = self.cfg["image_dir"]
        self.mask_dir  = self.cfg["mask_dir"]

        # mapping: grayscale_value(int) -> class_index(0-5)
        self.mapping      = {int(k): int(v) for k, v in self.cfg["mapping"].items()}
        self.num_classes  = int(self.cfg["num_classes"])
        self.ignore_index = self.cfg.get("ignore_index", 254)

        # Support optional file_list (for datasets split into groups))
        file_list_path = self.cfg.get("file_list", None)

        if file_list_path:
            # Load only the specific files listed in file_list
            with open(file_list_path) as f:
                allowed = set(json.load(f))
            self.masks  = sorted([f for f in os.listdir(self.mask_dir)
                                if f in allowed])
            stems       = {os.path.splitext(f)[0] for f in self.masks}
            self.images = sorted([f for f in os.listdir(self.image_dir)
                                if os.path.splitext(f)[0] in stems
                                and not f.startswith(".")])
        else:
            # No file_list = load everything (normal behavior for future labs)
            self.images = sorted([f for f in os.listdir(self.image_dir) if not f.startswith(".")])
            self.masks  = sorted([f for f in os.listdir(self.mask_dir)  if not f.startswith(".")])

        assert len(self.images) == len(self.masks), \
            f"{self.name}: {len(self.images)} images vs {len(self.masks)} masks — must match!"

        print(f"Loaded: {self.name} — {len(self.images)} images")

    def remap_mask(self, mask_np):
        """
        Convert grayscale pixel values -> class indices.
        Unknown values -> ignore_index (will be skipped in training).
        """
        remapped = np.full(mask_np.shape, self.ignore_index, dtype=np.int64)

        for gray_val, class_idx in self.mapping.items():
            remapped[mask_np == gray_val] = class_idx

        # Warn if unexpected values found (helps catch config errors)
        unknown = set(np.unique(mask_np).tolist()) - set(self.mapping.keys())
        if unknown:
            pass

        return remapped

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir,  self.masks[idx])

        img  = np.array(Image.open(img_path))
        mask = np.array(Image.open(mask_path))

        # --- Handle image channels ---
        # Some labs export RGB images, some grayscale
        # We always convert to single channel (grayscale) for CT
        if img.ndim == 3 and img.shape[2] == 4:
            img = img[:, :, :3]  # drop alpha channel first
        if img.ndim == 3:
            # RGB -> grayscale (weighted conversion)
            img = np.dot(img[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)
        # Now img is always (H, W)

        # --- Handle mask channels ---
        if mask.ndim == 3:
            mask = mask[:, :, 0]  # take first channel
        # Now mask is always (H, W)

        # Convert mask grayscale values -> class indices
        # --- FIX IMAGEJ CROPPING ERRORS ---
        H_img,  W_img  = img.shape
        H_mask, W_mask = mask.shape
        if H_img != H_mask or W_img != W_mask:
            new_mask = np.full((H_img, W_img), self.ignore_index, dtype=mask.dtype)
            paste_h = min(H_img, H_mask)
            paste_w = min(W_img, W_mask)
            new_mask[:paste_h, :paste_w] = mask[:paste_h, :paste_w]
            mask = new_mask
        # ----------------------------------
        mask = self.remap_mask(mask)

        # Image -> tensor [1, H, W] normalized [0, 1]
        img_t = torch.from_numpy(img.copy()).float().unsqueeze(0)
        # Z-score standardization on non-zero tissue pixels
        valid_pixels = img_t[img_t > 0]
        if len(valid_pixels) > 0:
            mean, std = valid_pixels.mean(), valid_pixels.std()
            if std > 1e-5:
                img_t = (img_t - mean) / std

        # Mask -> tensor [H, W] as int64
        mask_t = torch.from_numpy(mask.copy()).long()

        return img_t, mask_t


# --- Test ---
# Load all datasets from config
all_datasets = []
for path in CONFIG_PATHS:
    if os.path.exists(path):
        all_datasets.append(LeafDataset(path))

# Combine them into a single dataset
from torch.utils.data import ConcatDataset
base_ds = ConcatDataset(all_datasets)

img_t, mask_t = base_ds[0]
print(f"Image: {img_t.shape} {img_t.dtype} | min={img_t.min():.2f} max={img_t.max():.2f}")
print(f"Mask:  {mask_t.shape} {mask_t.dtype}")

Loaded: devin1_no_bse — 272 images
Loaded: devin1_with_bse — 56 images
Loaded: devin2 — 24 images
Loaded: devin3 — 7 images
Loaded: jg_laca — 8 images
Loaded: jg_mag — 12 images
Loaded: lf_arab — 55 images
Loaded: oak_ce — 18 images
Loaded: oak_cf — 18 images
Loaded: oak_cr — 18 images
Loaded: oak_ob — 18 images
Loaded: oak_ru — 18 images
Loaded: oak_su — 18 images
Loaded: olive_d4 — 6 images
Loaded: olive_d5 — 6 images
Loaded: olive_r1 — 6 images
Loaded: olive_r2 — 2 images
Loaded: olive_w4 — 6 images
Loaded: olive_w5 — 6 images
Loaded: st_pinus_aa — 6 images
Loaded: st_pinus_lo1 — 6 images
Loaded: st_pinus_lo2 — 6 images
Loaded: st_pinus_palus — 6 images
Loaded: st_pinus_pb — 6 images
Loaded: st_pinus_pc — 6 images
Loaded: st_pinus_pd — 6 images
Loaded: st_pinus_pe1 — 6 images
Loaded: st_pinus_pe2 — 7 images
Loaded: st_pinus_pf1 — 6 images
Loaded: st_pinus_pf3 — 5 images
Loaded: st_pinus_pg — 6 images
Loaded: st_pinus_ph — 6 images
Loaded: st_pinus_pinaster — 6 images
Loaded: st_pinu

---
## Step 6: PatchDataset

Splits large images into fixed-size patches for training.
- Covers 100% of every image (grid-based)
- Shuffled each epoch
- Augmentation applied during training only
- Works with ANY image size


In [7]:
class PatchDataset(Dataset):
    """
    Wraps LeafDataset and returns fixed-size patches.
    Works with any image size — pads if image is smaller than patch.
    """

    def __init__(self, base_dataset, patch_size=256, stride=128,
                 drop_background_only=True, augment=False):
        self.base   = base_dataset
        self.patch  = int(patch_size)
        self.stride = int(stride)
        self.drop_background_only = drop_background_only
        self.augment = augment
        self.patch_index = []

        # Augmentation pipeline (only applied when augment=True)
        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=50, p=0.5),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
            A.GaussNoise(p=0.2),
        ])

        self._build_index()

    def _grid_positions(self, H, W):
        """Generate grid of top-left patch positions covering entire image."""
        tops  = list(range(0, max(1, H - self.patch + 1), self.stride))
        lefts = list(range(0, max(1, W - self.patch + 1), self.stride))

        # Always include last position to cover image border
        if not tops  or tops[-1]  != max(0, H - self.patch):
            tops.append(max(0, H - self.patch))
        if not lefts or lefts[-1] != max(0, W - self.patch):
            lefts.append(max(0, W - self.patch))

        return tops, lefts

    def _build_index(self):
        self.patch_index = []
        for img_i in range(len(self.base)):
            img, mask = self.base[img_i]
            _, H, W = img.shape
            tops, lefts = self._grid_positions(H, W)
            for t in tops:
                for l in lefts:
                    # --- RARE CLASS OVERSAMPLING ---
                    mask_p = mask[t:min(t + self.patch, H), l:min(l + self.patch, W)]
                    uniq = set(mask_p.flatten().tolist())
                    
                    if self.drop_background_only:
                        # Drop patches that are ONLY background (class 0)
                        if len(uniq) == 1 and 0 in uniq:
                            continue
                            
                    # Multiply patches containing Vascular (4) or BSE (5) 
                    # 4x duplication physically oversamples the rare structures!
                    if 4 in uniq or 5 in uniq:
                        self.patch_index.extend([(img_i, t, l)] * 4)
                    else:
                        self.patch_index.append((img_i, t, l))
        print(f"Total valid patches: {len(self.patch_index)}")

    def shuffle_patches(self, seed=None):
        """Call before each epoch to randomize patch order."""
        if seed is not None:
            random.seed(seed)
        random.shuffle(self.patch_index)

    def __len__(self):
        return len(self.patch_index)

    def __getitem__(self, idx):
        img_i, t, l = self.patch_index[idx]
        img, mask = self.base[img_i]
        
        # --- DYNAMIC ISOLATION LOGIC (IGNORE SECONDARY LEAVES) ---
        # Find the mathematical bounding box of the annotated leaf
        annotated_coords = torch.nonzero(mask > 0)
        if len(annotated_coords) > 0:
            min_y = annotated_coords[:, 0].min().item()
            max_y = annotated_coords[:, 0].max().item()
            padding = 100  # pixels of acceptable margin around the main leaf
            
            # Create a blindfold mask
            # Everything outside the padded bounding box becomes 255 (Ignore)
            blindfold = torch.ones_like(mask, dtype=torch.bool)
            blindfold[max(0, min_y - padding):min(mask.shape[0], max_y + padding), :] = False
            
            # Only set unannotated background (0) to Ignore (IGNORE_INDEX). 
            # This protects accidental cropped annotations.
            mask[(mask == 0) & blindfold] = IGNORE_INDEX
        # --------------------------------------------------------
        
        _, H, W = img.shape

        # Crop patch
        img_p  = img[:, t:min(t + self.patch, H), l:min(l + self.patch, W)]
        mask_p = mask[t:min(t + self.patch, H), l:min(l + self.patch, W)]

        # Pad if patch is smaller than PATCH_SIZE
        # (happens when image is smaller than patch, or at borders)
        pad_h = self.patch - img_p.shape[1]
        pad_w = self.patch - img_p.shape[2]
        if pad_h > 0 or pad_w > 0:
            img_p  = F.pad(img_p,  (0, pad_w, 0, pad_h), value=0.0)
            pad_val = getattr(self.base.dataset if hasattr(self.base, 'dataset') else self.base, 'ignore_index', 254)
            mask_p = F.pad(mask_p, (0, pad_w, 0, pad_h), value=pad_val)

        # Apply augmentation (training only)
        if self.augment:
            img_np  = img_p.squeeze(0).numpy()
            mask_np = mask_p.numpy().astype(np.int32)
            # Albumentations expects HxWxC for images, but ours is currently HxW.
            # To satisfy A.Compose without dimension errors, we add a dummy channel.
            aug_out = self.aug(image=img_np[..., np.newaxis], mask=mask_np)
            # Remove the dummy channel and add the PyTorch channel dim back [1, H, W]
            img_p   = torch.from_numpy(aug_out['image'].squeeze(-1)).unsqueeze(0)
            mask_p  = torch.from_numpy(aug_out['mask']).long()

        return img_p, mask_p

---
## Step 7: Build DataLoader

In [8]:
# Split full images into train/val BEFORE making patches
# This prevents data leakage (same image in both train and val)
total    = len(base_ds)
val_size = int(total * VAL_FRAC)
trn_size = total - val_size

train_base, val_base = random_split(
    base_ds, [trn_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"Train images: {len(train_base)}")
print(f"Val images:   {len(val_base)}")

# Train: augmentation ON | Val: augmentation OFF
train_patch_ds = PatchDataset(train_base, PATCH_SIZE, STRIDE,
                               drop_background_only=True, augment=True)
val_patch_ds   = PatchDataset(val_base,   PATCH_SIZE, STRIDE,
                               drop_background_only=False, augment=False)

print(f"\nTrain patches: {len(train_patch_ds)}")
print(f"Val patches:   {len(val_patch_ds)}")

# NERSC-optimized DataLoaders
train_loader = DataLoader(train_patch_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

val_loader   = DataLoader(val_patch_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

# Verify
x, y = next(iter(train_loader))
print(f"\nBatch image: {x.shape} {x.dtype}")
print(f"Batch mask:  {y.shape} {y.dtype}")
print(f"Mask unique: {torch.unique(y).tolist()}")

Train images: 674
Val images:   74
Total valid patches: 106411
Total valid patches: 12935

Train patches: 106411
Val patches:   12935

Batch image: torch.Size([32, 1, 320, 320]) torch.float32
Batch mask:  torch.Size([32, 320, 320]) torch.int64
Mask unique: [0, 1, 3, 4, 254]


---
## Step 7.5: (Optional) Pre-trained Weights on NERSC Compute Nodes
NERSC compute nodes do not have internet access. If you set `encoder_weights="imagenet"` in Step 8, it will crash trying to download them.

**To fix this and get much higher accuracy:**
1. Open a terminal on a **Login Node** (which has internet).
2. Run these commands to download the weights to your scratch space:

-efficientnet-b4

```bash
export TORCH_HOME=/pscratch/sd/w/worasit/torch_cache
mkdir -p $TORCH_HOME
python3 -c "import segmentation_models_pytorch as smp; smp.UnetPlusPlus('efficientnet-b4', encoder_weights='imagenet')"
```

-timm-resnest101

```bash
mkdir -p /pscratch/sd/w/worasit/torch_cache/hub/checkpoints/ && wget -O /pscratch/sd/w/worasit/torch_cache/hub/checkpoints/resnest101-22405ba7.pth https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-resnest/resnest101-22405ba7.pth
```

3. In this notebook, add this line at the very top (Step 1):
```python
import os
os.environ['TORCH_HOME'] = '/pscratch/sd/w/worasit/torch_cache'
os.environ['HF_HUB_OFFLINE'] = '1'  # Crucial: prevents huggingface from pinging home and hanging!
```
4. Change `encoder_weights=None` to `encoder_weights="imagenet"` in Step 8. It will now load from the offline cache!

5. Change  `encoder_name="timm-resnest101e"` or `"efficientnet-b4"`

---
## Step 8: Model Setup

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# encoder_weights="imagenet" uses your new HuggingFace token to download weights automatically!
# in_channels=1 because CT images are grayscale (SMP handles the 3->1 channel sum for us)
model = smp.Unet(
    encoder_name="resnet101",
    encoder_weights="imagenet",
    in_channels=1,
    classes=NUM_CLASSES,
)
model = model.to(device)

# Hybrid loss: Focal (handles class imbalance) + Jaccard (shape quality)
# Same approach as original notebook but in multiclass mode
# Explicit Class Weights
# Classes: [Background(0), Epidermis(1), Vascular(2), Mesophyll(3), Air(4)]
# Penalize mistakes on rare Vascular/BSE tissues up to 4x harder than Background/Air!
class_weights = torch.tensor([0.5, 1.0, 2.0, 1.0, 1.0]).to(device)

# Provide weights manually inside criterion instead of via `alpha=` which crashes in SMP multiclass
loss_focal   = smp.losses.FocalLoss(mode="multiclass", ignore_index=IGNORE_INDEX)
loss_jaccard = smp.losses.JaccardLoss(mode="multiclass", classes=NUM_CLASSES)

def criterion(pred, mask):
    valid_mask = (mask != IGNORE_INDEX)
    
    # 1. Focal Loss (We apply the weights physically to the final loss scalar)
    focal_loss_raw = loss_focal(pred, mask)
    
    # Jaccard Loss doesn't always handle ignore_index well dynamically, so we flatten and filter
    b, c, h, w = pred.shape
    pred_flat = pred.permute(0, 2, 3, 1).reshape(-1, c)
    mask_flat = mask.view(-1)
    
    valid_idx = (mask_flat != IGNORE_INDEX)
    if valid_idx.sum() > 0:
        pred_valid = pred_flat[valid_idx]
        mask_valid = mask_flat[valid_idx]
        
        # Create a weight tensor for the valid pixels
        pixel_weights = class_weights[mask_valid]
        
        # Calculate Jaccard
        jaccard_loss = loss_jaccard(pred_valid, mask_valid)
        
        # Calculate CrossEntropy manually to allow clean pixel-wise weighting
        ce_loss = torch.nn.functional.cross_entropy(pred_valid, mask_valid, reduction='none')
        weighted_ce = (ce_loss * pixel_weights).mean()
    else:
        jaccard_loss = torch.tensor(0.0, device=device)
        weighted_ce = torch.tensor(0.0, device=device)

    # Hybrid Loss: 0.5 * Weighted CE + 0.5 * Jaccard
    # Note: We swap out SMP's FocalLoss for standard explicitly weighted Cross Entropy 
    #       because SMP's Focal alpha tensor broadcasting is globally broken for 2D batching.
    return 0.5 * weighted_ce + 0.5 * jaccard_loss

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready on: {device}")
print(f"Trainable parameters: {n_params/1e6:.1f}M")

/pscratch/sd/w/worasit/leafseg_venv/lib/python3.9/site-packages/segmentation_models_pytorch/encoders/__init__.py:136: UserWarning: Error loading resnet101 `imagenet` weights from Hugging Face Hub, trying loading from original url...
  warnings.warn(message, UserWarning)


Model ready on: cuda
Trainable parameters: 51.5M


---
## Step 9: Training Loop

In [10]:
BEST_MODEL_PATH = os.path.join(MODEL_DIR, "best_model.pth")
LAST_CHECKPOINT = os.path.join(MODEL_DIR, "last_checkpoint.pth")

best_val_loss  = float('inf')
epochs_no_improve = 0
history = []

print(f"Training started — best model -> {BEST_MODEL_PATH}")
print(f"Last checkpoint -> {LAST_CHECKPOINT}")
print("=" * 70)

# Resume from checkpoint - try LAST first, fall back to BEST
checkpoint_path = None
if os.path.exists(LAST_CHECKPOINT):
    checkpoint_path = LAST_CHECKPOINT
    print(f"Found last checkpoint")
elif os.path.exists(BEST_MODEL_PATH):
    checkpoint_path = BEST_MODEL_PATH
    print(f"Found best model, using as starting point")

if checkpoint_path:
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # Safely load scheduler if it exists in this checkpoint
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    # OVERRIDE: Force optimizer to use our new manual LR from Step 4
    for param_group in optimizer.param_groups:
        param_group['lr'] = LR
    start_epoch = checkpoint['epoch']  # Resume directly from saved epoch
    best_val_loss = checkpoint.get('best_val_loss', checkpoint.get('val_loss', float('inf')))
    print(f"Resumed from epoch {checkpoint['epoch']}, best_val_loss={best_val_loss:.4f}")
else:
    start_epoch = 0
    best_val_loss = float('inf')
    print("Starting fresh")

for epoch in range(start_epoch, NUM_EPOCHS):
    
    # Shuffle patches each epoch
    train_patch_ds.shuffle_patches(seed=epoch)
    
    # ===== TRAIN =====
    model.train()
    train_loss = 0.0
    
    for x, y in tqdm(train_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS} [Train]", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast('cuda', dtype=torch.bfloat16):
            pred = model(x)
            loss = criterion(pred, y)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train = train_loss / len(train_loader)
    
    # ===== VALIDATE =====
    model.eval()
    val_loss = 0.0
    epochs_valid_batches = 0
    
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS} [Val]", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            with autocast('cuda', dtype=torch.bfloat16):
                pred = model(x)
                loss = criterion(pred, y)
            
            import math
            loss_val = loss.item()
            if not math.isnan(loss_val) and not math.isinf(loss_val):
                val_loss += loss_val
                epochs_valid_batches += 1
    
    if epochs_valid_batches > 0:
        avg_val = val_loss / epochs_valid_batches
    else:
        avg_val = float('inf')
    scheduler.step(avg_val)
    
    # ===== SAVE LAST CHECKPOINT (every epoch) =====
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': avg_train,
        'val_loss': avg_val,
        'best_val_loss': best_val_loss,
        'num_classes': NUM_CLASSES,
        'patch_size': PATCH_SIZE,
        'configs': CONFIG_PATHS,
    }, LAST_CHECKPOINT)
    
    # ===== SAVE BEST MODEL =====
    saved_str = ""
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': avg_train,
            'val_loss': avg_val,
            'num_classes': NUM_CLASSES,
            'patch_size': PATCH_SIZE,
            'configs': CONFIG_PATHS,
        }, BEST_MODEL_PATH)
        saved_str = " <- saved best"
    else:
        epochs_no_improve += 1
    
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.1e}{saved_str}")
    
    history.append({'epoch': epoch+1, 'train_loss': avg_train, 
                    'val_loss': avg_val, 'lr': optimizer.param_groups[0]['lr']})
    
    # Early stopping
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    # Periodic GPU memory cleanup
    if (epoch + 1) % 20 == 0:
        torch.cuda.empty_cache()
        gc.collect()

# Save loss history
hist_df = pd.DataFrame(history)
hist_df.to_csv(os.path.join(LOG_DIR, "training_history.csv"), index=False)

print("=" * 70)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")
print(f"Best model: {BEST_MODEL_PATH}")

Training started — best model -> /pscratch/sd/w/worasit/outputs/models_Unet_ResNet101/best_model.pth
Last checkpoint -> /pscratch/sd/w/worasit/outputs/models_Unet_ResNet101/last_checkpoint.pth
Found last checkpoint
Resumed from epoch 50, best_val_loss=0.4345


KeyboardInterrupt: 

---
## Step 10: Evaluate Best Model
Per-class metrics: IoU, Dice, Precision, Recall

In [11]:
CLASS_NAMES = ["Background", "Epidermis", "Vascular_Region", "Mesophyll", "Air_Space"]

# Load best checkpoint
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model_eval = smp.Unet(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=1,
    classes=NUM_CLASSES,
)
model_eval.load_state_dict(checkpoint['model_state_dict'])
model_eval = model_eval.to(device)
model_eval.eval()

print(f"Loaded model from epoch {checkpoint['epoch']} (val loss: {checkpoint['val_loss']:.4f})")

# Accumulate metrics
tp = torch.zeros(NUM_CLASSES)
fp = torch.zeros(NUM_CLASSES)
fn = torch.zeros(NUM_CLASSES)
present = torch.zeros(NUM_CLASSES)  # track which classes appear

with torch.no_grad():
    for x, y in tqdm(val_loader, desc="Evaluating"):
        x = x.to(device)
        y = y.to(device)

        pred     = model_eval(x)
        pred_cls = torch.argmax(pred, dim=1)

        # Ignore padding/unknown pixels
        valid = (y != IGNORE_INDEX)

        for c in range(NUM_CLASSES):
            pred_c = (pred_cls == c) & valid
            true_c = (y == c)        & valid

            if true_c.sum() > 0:
                tp[c] += (pred_c & true_c).sum().float().cpu()
                fp[c] += (pred_c & ~true_c).sum().float().cpu()
                fn[c] += (~pred_c & true_c).sum().float().cpu()
                present[c] += 1

precision = tp / (tp + fp + 1e-7)
recall    = tp / (tp + fn + 1e-7)
dice      = 2 * tp / (2 * tp + fp + fn + 1e-7)
iou       = tp / (tp + fp + fn + 1e-7)

rows = []
for c in range(NUM_CLASSES):
    if present[c] > 0:
        rows.append({
            'Class':     CLASS_NAMES[c],
            'IoU':       round(iou[c].item(), 4),
            'Dice':      round(dice[c].item(), 4),
            'Precision': round(precision[c].item(), 4),
            'Recall':    round(recall[c].item(), 4),
        })
    else:
        # Class not present in this dataset (e.g. BSE)
        rows.append({'Class': CLASS_NAMES[c], 'IoU': 'N/A',
                     'Dice': 'N/A', 'Precision': 'N/A', 'Recall': 'N/A'})

modeldata = pd.DataFrame(rows)
display(modeldata)

# Mean IoU (only classes present)
numeric_iou = [r['IoU'] for r in rows if r['IoU'] != 'N/A']
print(f"\nMean IoU (present classes):  {np.mean(numeric_iou):.4f}")
print(f"Mean Dice (present classes): {np.mean([r['Dice'] for r in rows if r['Dice'] != 'N/A']):.4f}")

metrics_csv = os.path.join(LOG_DIR, "validation_metrics.csv")
modeldata.to_csv(metrics_csv, index=False)
print(f"\nMetrics saved: {metrics_csv}")

Loaded model from epoch 18 (val loss: 0.4345)


Evaluating: 100%|██████████| 405/405 [03:28<00:00,  1.94it/s]


,Class,IoU,Dice,Precision,Recall
0,Background,0.8938,0.9439,0.9267,0.9618
1,Epidermis,0.7230,0.8392,0.8628,0.8169
2,Vascular_Region,0.7123,0.8320,0.9253,0.7557
3,Mesophyll,0.7910,0.8833,0.8583,0.9098
4,Air_Space,0.7580,0.8623,0.8881,0.8380



Mean IoU (present classes):  0.7756
Mean Dice (present classes): 0.8721

Metrics saved: /pscratch/sd/w/worasit/outputs/logs_Unet_ResNet101/validation_metrics.csv


---
## Step 11: Inference
Run trained model on new images.

In [ ]:
# Define paths
SCRATCH = "/pscratch/sd/w/worasit"
OUTPUT_DIR = f"{SCRATCH}/outputs"
MODEL_DIR = f"{OUTPUT_DIR}/models_Unet_ResNet101"
BEST_MODEL_PATH = f"{MODEL_DIR}/best_model.pth"

INFERENCE_DIR = "/pscratch/sd/w/worasit/datasets/devin1/images" # Change for new tesst set
INFERENCE_OUT = "/pscratch/sd/w/worasit/outputs/inference_masks"
INFERENCE_CSV = "/pscratch/sd/w/worasit/outputs/inference_results.csv"

os.makedirs(INFERENCE_OUT, exist_ok=True)

# Class names
CLASS_NAMES = ["Background", "Epidermis", "Vascular_Region", "Mesophyll", "Air_Space"]

# Load best checkpoint
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model_eval = smp.Unet(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=1,
    classes=NUM_CLASSES,
)
model_eval.load_state_dict(checkpoint['model_state_dict'])
model_eval = model_eval.to(device)
model_eval.eval()

print(f"Loaded model from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

# ============================================================
# Find images
img_files = sorted([f for f in os.listdir(INFERENCE_DIR)
                    if f.lower().endswith(('.png','.tif','.tiff','.jpg','.jpeg'))])

print(f"Found {len(img_files)} images")

results = []

for fname in tqdm(img_files, desc="Inference"):
    img = np.array(Image.open(os.path.join(INFERENCE_DIR, fname)))
    
    # Handle RGB → grayscale (same as training)
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    if img.ndim == 3:
        img = np.dot(img[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8).copy()
    
    img_t = torch.from_numpy(img.copy()).float().unsqueeze(0).unsqueeze(0)
    # Z-score standardization on non-zero tissue pixels (MUST MATCH TRAINING!)
    valid_pixels = img_t[img_t > 0]
    if len(valid_pixels) > 0:
        mean, std = valid_pixels.mean(), valid_pixels.std()
        if std > 1e-5:
            img_t = (img_t - mean) / std

    # Pad to multiple of 32
    h, w = img.shape
    pad_h = (32 - h % 32) % 32
    pad_w = (32 - w % 32) % 32
    if pad_h > 0 or pad_w > 0:
        img_t = torch.nn.functional.pad(img_t, (0, pad_w, 0, pad_h), mode='reflect')


    img_t = img_t.to(device)
    
    # --- SLIDING WINDOW INFERENCE TO PREVENT OOM ---
    # SegFormer Self-Attention is O(N^2) memory. Massive 2560x2560 images will instantly OOM 40GB A100s.
    # We must chunk it in pieces that match our training size and average the overlaps.
    patch_size = 320
    stride = 160
    
    _, _, H_t, W_t = img_t.shape
    pred_prob_accum = torch.zeros((1, NUM_CLASSES, H_t, W_t), device=device, dtype=torch.float32)
    count_accum     = torch.zeros((1, 1, H_t, W_t), device=device, dtype=torch.float32)
    
    with torch.no_grad():
        with autocast('cuda', dtype=torch.bfloat16):
            # Iterate through grid
            for py in range(0, H_t, stride):
                for px in range(0, W_t, stride):
                    # Bounds
                    y1 = min(py, H_t - patch_size)
                    y2 = y1 + patch_size
                    x1 = min(px, W_t - patch_size)
                    x2 = x1 + patch_size
                    
                    # Extract patch
                    patch = img_t[:, :, y1:y2, x1:x2]
                    
                    # 1. Original
                    p_orig = model_eval(patch).softmax(dim=1)
                    
                    # 2. Horizontal Flip
                    p_hflip = model_eval(torch.flip(patch, dims=[3]))
                    p_hflip = torch.flip(p_hflip.softmax(dim=1), dims=[3])
                    
                    # 3. Vertical Flip
                    p_vflip = model_eval(torch.flip(patch, dims=[2]))
                    p_vflip = torch.flip(p_vflip.softmax(dim=1), dims=[2])
                    
                    p_avg = (p_orig + p_hflip + p_vflip) / 3.0
                    
                    # Accumulate
                    pred_prob_accum[:, :, y1:y2, x1:x2] += p_avg
                    count_accum[:, :, y1:y2, x1:x2] += 1.0
                    
    # Average the overlaps
    pred_prob = pred_prob_accum / count_accum
    pred_cls = torch.argmax(pred_prob, dim=1).squeeze().cpu().numpy()
            
    # Crop back to original size
    if pad_h > 0 or pad_w > 0:
        pred_cls = pred_cls[:h, :w]

    # Save prediction mask
    stem = os.path.splitext(fname)[0]
    out_path = os.path.join(INFERENCE_OUT, f"{stem}_pred.png")
    Image.fromarray(pred_cls.astype(np.uint8)).save(out_path)
    
    # Calculate tissue fractions
    total = pred_cls.size
    row = {'filename': fname}
    for c, name in enumerate(CLASS_NAMES):
        count = int(np.sum(pred_cls == c))
        row[f"{name}_pixels"] = count
        row[f"{name}_fraction"] = round(count / total, 6)
    row['porosity'] = row['Air_Space_fraction']
    results.append(row)

results_df = pd.DataFrame(results)
results_df.to_csv(INFERENCE_CSV, index=False)

print(f"\nDone!")
print(f"Masks: {INFERENCE_OUT}")
print(f"Data:  {INFERENCE_CSV}")
display(results_df.head())

---
# Pipeline Complete!

```
/pscratch/sd/w/worasit/outputs/
├── models/best_model.pth          <- best checkpoint
├── logs/training_history.csv      <- loss per epoch  
├── logs/validation_metrics.csv    <- IoU, Dice per class
└── inference_masks/*_pred.png     <- predicted masks
```

---
## Check Inference Image

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Load one prediction mask
import glob
mask_files = sorted(glob.glob('/pscratch/sd/w/worasit/outputs/inference_masks/*_pred.png'))
if len(mask_files) > 0:
    print(f"Loading: {mask_files[99]}")
    mask = np.array(Image.open(mask_files[99]))
else:
    print('No prediction masks found!')
    mask = np.zeros((10,10))  # Dummy fallback

# Check what's actually in it
print("Mask shape:", mask.shape)
print("Unique values:", np.unique(mask))
print("Value counts:")
for val in np.unique(mask):
    count = np.sum(mask == val)
    pct = 100 * count / mask.size
    print(f"  Class {val}: {count:,} pixels ({pct:.1f}%)")

# Visualize with colormap
plt.figure(figsize=(10, 10))
plt.imshow(mask, cmap='tab10', vmin=0, vmax=5)
plt.colorbar(label='Class')
plt.title('Predicted Segmentation')
plt.show()

## Check Fraction

In [ ]:
import pandas as pd

df = pd.read_csv('/pscratch/sd/w/worasit/outputs/inference_results.csv')

# Check first image
fraction_cols = [col for col in df.columns if 'fraction' in col]
print(df.iloc[2][fraction_cols])